# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Answer:**

I chose Random Forest because it can capture nonlinear relationships between the features and the target without requiring feature scaling. It can handle features with different numerical magnitudes and provides feature importance, which helps with interpretation. It also provides a stronger model to compare against the simpler baseline.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [1]:
import pandas as pd

df = pd.read_csv("/content/content_refresh_anonymized.csv")
df.head()


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [11]:
df.shape

(30000, 44)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 44 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   content_id              30000 non-null  object 
 1   client_id               30000 non-null  object 
 2   search_volume           27532 non-null  float64
 3   competition             27532 non-null  float64
 4   competition_level       27390 non-null  object 
 5   cpc                     27532 non-null  float64
 6   content_type            30000 non-null  object 
 7   main_intent             27626 non-null  object 
 8   word_count              22301 non-null  float64
 9   char_count              22301 non-null  float64
 10  provider_used           8562 non-null   object 
 11  model_used              24267 non-null  object 
 12  impressions_90d         30000 non-null  int64  
 13  clicks_90d              30000 non-null  int64  
 14  pageviews_90d           30000 non-null

In [6]:
feature_cols = ["ctr", "avg_position", "engagement_rate"]

X = df[feature_cols]
y = (df["trend_direction"] == "down").astype(int)


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Answer:**

I use an 80/20 train-test split with stratification to preserve the proportion of declining and non-declining examples in both sets. The test set is kept separate during training so that model performance can be evaluated on unseen data.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)
print(X_train.shape)
print(X_test.shape)
print(y_train.value_counts())
print(y_test.value_counts())

(24000, 3)
(6000, 3)
trend_direction
1    13010
0    10990
Name: count, dtype: int64
trend_direction
1    3252
0    2748
Name: count, dtype: int64


In [10]:
y.value_counts()

,count
trend_direction,
1,16262
0,13738


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

**Answer:**

I use a Random Forest classifier to predict whether a page is declining. Random Forest is suitable because it can model nonlinear relationships between the input signals without requiring feature scaling.

The model is evaluated on the held-out test set. To compare it with the Week-4 rule-based baseline, the baseline ranking is evaluated using the same test rows and classification metrics. The comparison focuses on precision, recall, and F1 rather than accuracy alone, because the main objective is identifying declining pages.

The Week-4 baseline ranks pages using a hand-designed action score based on content age, search impressions, and trend direction.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [12]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score
import pandas as pd

# Train Random Forest
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced"
)

rf_model.fit(X_train, y_train)

# Predictions on unseen test data
y_pred = rf_model.predict(X_test)

# Random Forest metrics
rf_precision = precision_score(y_test, y_pred)
rf_recall = recall_score(y_test, y_pred)
rf_f1 = f1_score(y_test, y_pred)

print("Random Forest:")
print(f"Precision: {rf_precision:.3f}")
print(f"Recall:    {rf_recall:.3f}")
print(f"F1:        {rf_f1:.3f}")

Random Forest:
Precision: 0.608
Recall:    0.637
F1:        0.622


In [15]:
from sklearn.metrics import precision_score, recall_score, f1_score

# Recreate the Week-4 baseline score
baseline_test = df.loc[X_test.index].copy()

baseline_test["baseline_score"] = (
    baseline_test["days_since_last_update"].astype(float)
    * baseline_test["impressions_90d"].astype(float)
    * baseline_test["trend_direction"].eq("down").map({True: 1.0, False: 0.5})
)

# Rank test-set pages by the Week-4 baseline score
baseline_test = baseline_test.sort_values(
    "baseline_score",
    ascending=False
)

# Top 20% are predicted as declining
cutoff = int(len(baseline_test) * 0.20)

baseline_test["baseline_pred"] = 0
baseline_test.iloc[:cutoff, baseline_test.columns.get_loc("baseline_pred")] = 1

# Put predictions back in the original test-set order
baseline_test = baseline_test.loc[X_test.index]

y_baseline = baseline_test["baseline_pred"]

# Evaluate baseline
baseline_precision = precision_score(y_test, y_baseline)
baseline_recall = recall_score(y_test, y_baseline)
baseline_f1 = f1_score(y_test, y_baseline)

print("Week-4 Baseline:")
print(f"Precision: {baseline_precision:.3f}")
print(f"Recall:    {baseline_recall:.3f}")
print(f"F1:        {baseline_f1:.3f}")

Week-4 Baseline:
Precision: 0.650
Recall:    0.240
F1:        0.350


In [16]:
comparison = pd.DataFrame({
    "Model": ["Week-4 Baseline", "Random Forest"],
    "Precision": [baseline_precision, rf_precision],
    "Recall": [baseline_recall, rf_recall],
    "F1": [baseline_f1, rf_f1]
})

print("\n=== MODEL COMPARISON ===")
print(comparison.to_string(index=False))


=== MODEL COMPARISON ===
          Model  Precision   Recall       F1
Week-4 Baseline   0.650000 0.239852 0.350404
  Random Forest   0.608338 0.637146 0.622409


The Random Forest achieved a precision of 0.608, recall of 0.637, and F1 score of 0.622 on the held-out test set.

The Week-4 rule-based baseline achieved a precision of 0.650, recall of 0.240, and F1 score of 0.350 on the same test rows.

The baseline has slightly higher precision, meaning its positive predictions are somewhat more selective. However, the Random Forest has substantially higher recall and a higher F1 score. This indicates that the Random Forest identifies more of the observed declining pages while maintaining a reasonable level of precision.

Overall, the Random Forest provides a stronger balance between precision and recall than the Week-4 baseline for this classification task.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

**Answer:**

I inspect the model's false positives and false negatives to understand where its predictions disagree with the observed labels. I also examine feature importance to identify which input signals the Random Forest relies on most.

The error analysis is used as decision-support rather than proof that a particular feature causes decline. The results indicate which signals are useful for the model's predictions and where additional investigation may be needed.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [13]:
# Build an error-analysis dataframe
error_df = X_test.copy()
error_df["actual"] = y_test.values
error_df["predicted"] = y_pred

# False positives: predicted declining, actually not declining
false_positives = error_df[
    (error_df["actual"] == 0) &
    (error_df["predicted"] == 1)
]

# False negatives: predicted not declining, actually declining
false_negatives = error_df[
    (error_df["actual"] == 1) &
    (error_df["predicted"] == 0)
]

print("=== ERROR ANALYSIS ===")
print(f"False positives: {len(false_positives)}")
print(f"False negatives: {len(false_negatives)}")

print("\n=== FEATURE IMPORTANCE ===")

importance_df = pd.DataFrame({
    "feature": feature_cols,
    "importance": rf_model.feature_importances_
}).sort_values("importance", ascending=False)

print(importance_df.to_string(index=False))

=== ERROR ANALYSIS ===
False positives: 1334
False negatives: 1180

=== FEATURE IMPORTANCE ===
        feature  importance
   avg_position    0.595183
            ctr    0.245252
engagement_rate    0.159565



The Random Forest produced 1,334 false positives and 1,180 false negatives on the test set. This shows that the model still makes errors in both directions: some pages are predicted as declining when their observed label is not declining, while some declining pages are missed.

The feature importance results show that avg_position was the strongest model signal, with an importance of 0.595. ctr contributed 0.245 and engagement_rate contributed 0.160. Therefore, the model relied most heavily on average search position when making its predictions.

These feature importance values describe model reliance rather than causation. Further investigation would be needed to determine why pages with different average positions are associated with the observed decline labels.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.